# Ollama

In [1]:
!pip cache purge --quiet

In [2]:
!pip install langchain==0.3.27 \
             langchain-ollama==0.3.10 \
             langchain-singlestore==1.0.0 \
             ollama==0.6.0 \
             singlestoredb==1.15.8 \
             sqlalchemy-singlestoredb==1.1.2 \
             --quiet

In [3]:
import logging
import ollama
import os

from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_singlestore.vectorstores import SingleStoreVectorStore
from urllib.parse import quote

In [4]:
logging.getLogger("httpx").setLevel(logging.ERROR)

In [5]:
EMBEDDING_MODEL = "all-minilm"
LLM_MODEL = "llama3"

In [6]:
ollama.pull(EMBEDDING_MODEL)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [7]:
ollama.pull(LLM_MODEL)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [8]:
company_info = {
    "AAPL": "Apple Inc. is a technology company known for the iPhone, iPad and Mac. It also offers services like iCloud and Apple Music.",
    "AMZN": "Amazon.com is an e-commerce giant with businesses in cloud computing (AWS), logistics and digital streaming.",
    "GOOG": "Google, a subsidiary of Alphabet Inc., is a technology company specializing in internet services, online advertising, search and AI.",
    "MSFT": "Microsoft develops software such as Windows and Office and is a leader in cloud computing with Azure."
}

In [9]:
documents = [
    Document(page_content = text, metadata = {"symbol": sym})
    for sym, text in company_info.items()
]

In [10]:
# Use embeddings
embeddings = OllamaEmbeddings(
    model = EMBEDDING_MODEL,
)

# Vector dimension is fixed per model, but can still check dynamically
dimensions = len(embeddings.embed_query("test"))

In [11]:
username = "admin"
password = os.environ.get("SINGLESTOREDB_PASSWORD")
host = os.environ.get("SINGLESTOREDB_HOST")
port = 3306
database = "ollama_db"

if not password:
    raise ValueError("Environment variable SINGLESTOREDB_PASSWORD is not set or is empty.")

if not host:
    raise ValueError("Environment variable SINGLESTOREDB_HOST is not set or is empty.")

problematic_chars = ["#", "@", "/", "?", "%"]
found = [c for c in problematic_chars if c in password]
if found:
    print(f"WARNING: Password contains character(s) {found} which may cause connection issues.")

my_connection_url = f"singlestoredb://{username}:{quote(password, safe='')}@{host}:{port}/{database}"

In [12]:
from sqlalchemy import *

db_connection = create_engine(my_connection_url)

In [13]:
try:
    with db_connection.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS company_knowledge;"))
except Exception as e:
    print(f"Error dropping table: {e}")
    raise

In [14]:
vector_store = SingleStoreVectorStore(
    embeddings,
    user = username,
    password = password,
    host = host,
    port = port,
    database = database,
    table_name = "company_knowledge",
    distance_strategy = "DOT_PRODUCT",
    use_vector_index = True,
    vector_size = dimensions
)

vector_store.add_documents(documents);

In [15]:
prompt = "What are the most popular consumer devices and services that Apple Inc. sells?"
documents = vector_store.similarity_search(prompt, k = 1)
data = documents[0].page_content
print(data)

Apple Inc. is a technology company known for the iPhone, iPad and Mac. It also offers services like iCloud and Apple Music.


In [16]:
output = ollama.generate(
    model = LLM_MODEL,
    prompt = f"Using this data: {data}. Respond to this prompt: {prompt}",
    options = {
        "temperature": 0
    }
)

print(output["response"])

Based on the provided data, the most popular consumer devices sold by Apple Inc. are:

1. iPhone
2. iPad
3. Mac (computers)

As for services, Apple Inc. offers:

1. iCloud (cloud storage and backup)
2. Apple Music (music streaming)

These products and services are some of the most well-known and widely used among consumers, making them the most popular offerings from Apple Inc.
